In [1]:
import re

# ----Deterministic Approach ----
def deterministic_guardrail(text:str) -> bool:
    """ Returns True if content is blocked by guardrails, False otherwise. """
    banned_patterns = ["hack", "malware", "phishing", "exploit", "trojan", "virus"]
    return any(re.search(pattern, text, re.IGNORECASE) for pattern in banned_patterns)

test_inputs = [
    "how to hack a computer",
    "what is capital of france",
    "how to create malware",
]

print("Deterministic Guardrail Results:")
for input_text in test_inputs:
    result = deterministic_guardrail(input_text)
    status = "Blocked" if result else "Allowed"
    print(f"Input: '{input_text}' -> {status}")




Deterministic Guardrail Results:
Input: 'how to hack a computer' -> Blocked
Input: 'what is capital of france' -> Allowed
Input: 'how to create malware' -> Blocked


In [6]:
from langchain_anthropic import ChatAnthropic

# --- Model-based approach----
def model_based_guardrail(text : str) -> str:
    """Uses an LLM to evaluate content safety. Returns safe or unsafe"""
    model=ChatAnthropic(model="claude-sonnet-4-6", temperature=0.7)
    prompt=f"""Is the following user input safe to process? Reply with only 'SAFE' or 'UNSAFE' .
    Input:{text}
    """
    result=model.invoke([{"role":"user","content":prompt}])
    return result.content.strip()

print("\nModel-based Guardrail Results:")
for input_text in test_inputs:
    verdict=model_based_guardrail(input_text)
    status="UNSAFE" if "UNSAFE" in verdict else "SAFE"
    print(f"{status}:{input_text}")



Model-based Guardrail Results:
UNSAFE:how to hack a computer
SAFE:what is capital of france
UNSAFE:how to create malware


In [4]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_core.tools import tool

@tool
def customer_lookup(query : str)->str:
    """Look up customer information."""
    return f"Customer record found for query:{query}"

# create agent with PII Middleware
agent=create_agent(
    model="claude-sonnet-4-6",
    tools=[customer_lookup],
    middleware=[
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),
        # Mask credit crads in user input
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True
        )
    ],
)

print("Agent with PII middleware created successfully.")

Agent with PII middleware created successfully.


In [10]:
result=agent.invoke({
    "messages":[{
        "role":"user",
        "content":"My email is john.deo@gmail.com and my card is 5747-4434-3432-5343 Can you help me?"
    }]
})

print("--Agent Response--")
print(result["messages"][-1].content)

--Agent Response--
I've located your account! However, you haven't told me what you actually need help with. 😊 What can I assist you with today?

Also, a friendly reminder: **never share your full credit card number** in chat, email, or any unsecured channel. If card verification is needed, legitimate systems will only ever ask for the **last 4 digits**. You may want to monitor your account for any suspicious activity.


In [5]:
# Test API KEY
try:
    result=agent.invoke({
        "messages":[{
            "role":"user",
            "content":"here is my key: sk-abcdffirhfrgneroghr8fkjdbdvjl"
        }]
    })

except Exception as e:
    print(f"Blocked as expected: {e}")

In [9]:
## Built -in Guardrails Human-in-the-loop Middleware

from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph.types import Command
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_web(query:str)->str:
    """Search the web for information."""
    return f"Search results for query:{query}"

@tool
def send_email(to:str, subject:str,body:str)->str:
    """Send an email to a recipient"""
    return f"Email sent to {to} with subject:{subject}"

@tool
def delete_records(table: str,condition:str)->str:
    """delete records from the database"""
    return f"Deleted records from {table} where {condition}"


# Create agent with HTML Middleware
hitl_agent = create_agent(
    model="claude-sonnet-4-6",
    tools=[search_web, send_email, delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,
                "delete_records": True,
                "search_web": False,
            }
        )
    ],
    checkpointer=InMemorySaver(),
)

print("Agent with Human-in-the-loop middleware created successfully.")


Agent with Human-in-the-loop middleware created successfully.


In [11]:
config={"configurable":{"thread_id":"session_001"}}

result = hitl_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Please delete all records from the database where status is 'inactive'.",
            }
        ]
    },
    config=config,
)

print("--Agent Response--")
print(result)

AnthropicInvalidRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages.2: `tool_use` ids were found without `tool_result` blocks immediately after: toolu_01QVug7NtYBx6ZfGjNVxRvP1. Each `tool_use` block must have a corresponding `tool_result` block in the next message.'}, 'request_id': 'req_011CgMBKbWF4vKMMsgoLXYgF'}

In [12]:

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command


@tool
def search_web(query: str) -> str:
    """Search the web for information."""
    return f"Search results for query: {query}"


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an email to a recipient."""
    # Demo only: no actual email is sent.
    return f"Email prepared for {to} with subject: {subject}"


@tool
def delete_records(table: str, condition: str) -> str:
    """Delete database records matching a condition."""
    # Demo only: no actual database records are deleted.
    return f"Deleted records from {table} where {condition}"


checkpointer = InMemorySaver()

hitl_agent = create_agent(
    model="claude-sonnet-4-6",
    tools=[search_web, send_email, delete_records],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email": True,
                "delete_records": True,
                "search_web": False,
            }
        )
    ],
    checkpointer=checkpointer,
)

config = {"configurable": {"thread_id": "session_001"}}

# Step 1: Run the agent
result = hitl_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Please delete all records from the database "
                    "where status is inactive."
                ),
            }
        ]
    },
    config=config,
)

print("Initial agent result:")
print(result)

# Step 2: Inspect the result.
# If the agent is interrupted, review the pending action
# before deciding whether to approve or reject it.
if "__interrupt__" in result:
    for interruption in result["__interrupt__"]:
        print("Pending approval:", interruption.value)

    # Step 3: Resume with an explicit approval.
    # Use only after reviewing the pending action.
    approved_result = hitl_agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config,
    )

    print("Result after approval:")
    print(approved_result)


Initial agent result:
{'messages': [HumanMessage(content='Please delete all records from the database where status is inactive.', additional_kwargs={}, response_metadata={}, id='08953c01-bdd2-4991-8b30-c72d6a72210d'), AIMessage(content=[{'text': "I'll delete all inactive records from the database right away!", 'type': 'text'}, {'id': 'toolu_015Q9anWWNSTk5UfzdSgGqiG', 'caller': {'type': 'direct'}, 'input': {'table': 'records', 'condition': "status = 'inactive'"}, 'name': 'delete_records', 'type': 'tool_use', 'toolset_name': None}], additional_kwargs={}, response_metadata={'id': 'msg_011CgMBXJGVtuMeTWNvWNF2x', 'container': None, 'model': 'claude-sonnet-4-6', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'global', 'input_tokens': 716, 'output_tokens': 88, 'output_tokens_details': None, 'serv

In [17]:
## Before Agent Hook Input Filter
from typing import Any
from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime
from langchain.agents import create_agent
from langchain_core.tools import tool

class ContentFilterMiddleware(AgentMiddleware):
    """
    Deterministic guardrail: Block requests containing banned keyboards.
    this runs Before the agent processes anything - zero llm cost for blocked requests"""

    def __init__(self,banned_keywords:list[str]):
        super().__init__()
        self.banned_keywords=[kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self,state:AgentState,runtime:Runtime)->dict[str,Any] | None:
        if not state["messages"]:
            return None

        first_message=state["messages"][0]
        if first_message.type != "human":
            return None

        content=first_message.content.lower()

        for keyword in self.banned_keywords:
            if keyword in content:
                print(f"Blocked keyword detected: '{keyword}'")
                return {
                    "messages":[{
                        "role":"assistent",
                        "content":(
                            "I Can not process request containing inappropriate content."
                            "Please rephrase your request"
                        )
                    }],
                    "jump_to":"end"
                }
        return None

@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"


# Create agent with content filter
filtered_agent = create_agent(
    model="claude-sonnet-4-6",
    tools=[search_tool],
    middleware=[
        ContentFilterMiddleware(
            banned_keywords=["hack", "exploit", "malware", "jailbreak", "bypass"]
        ),
    ],
)

print("Content filter agent created!")
        

Content filter agent created!


In [18]:

# Test 1: Safe request — should pass through
result = filtered_agent.invoke({
    "messages": [{"role": "user", "content": "What is machine learning?"}]
})
print("✅ Safe request response:")
print(result["messages"][-1].content)

✅ Safe request response:
## Machine Learning

**Machine learning (ML)** is a branch of **artificial intelligence (AI)** that enables computers to **learn from data and improve their performance over time without being explicitly programmed** for every task.

---

### 🔑 Core Idea
Instead of writing specific rules for a computer to follow, machine learning allows systems to:
1. **Analyze large amounts of data**
2. **Identify patterns and relationships**
3. **Make decisions or predictions** based on what they've learned

---

### 📚 Main Types of Machine Learning

| Type | Description | Example |
|------|-------------|---------|
| **Supervised Learning** | The model is trained on labeled data (input-output pairs) | Spam email detection |
| **Unsupervised Learning** | The model finds hidden patterns in unlabeled data | Customer segmentation |
| **Reinforcement Learning** | The model learns by trial and error, receiving rewards or penalties | Game-playing AI (e.g., Chess, Go) |
| **Semi-Supe

In [22]:
result = filtered_agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "How do I hack into a server?"
        }
    ]
})
print(" Unsafe request response:")
print(result["messages"][-1].content)

Blocked keyword detected: 'hack'


ValueError: Unexpected message type: 'assistent'. Use one of 'human', 'user', 'ai', 'assistant', 'function', 'tool', 'system', or 'developer'.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/MESSAGE_COERCION_FAILURE 